In [32]:
from utils.llm_api.client import GPT4AllClient
import os

In [33]:
# generate query that should be used in LLM for evaluating generated code and reference code
query = """
    You are code quality judge. You have been given two pieces of code. One is the reference code and the other is the generated code. You have to evaluate the generated code based on the reference code. 
    You have to evaluate the generated code based on the following criteria:
    1. Correctness: The generated code should produce the same output as the reference code.
    2. Readability: The generated code should be easy to read and understand.
    
    The reference code is: %s
    The generated code is: %s
    
    You must provide only single float value as the output. Value should be between 0 and 1. Do not provide any additional information.
"""

In [34]:
MODEL_NAME_JUDGE = "Llama 3.2 3B Instruct"
client_corrector = GPT4AllClient(model_name=MODEL_NAME_JUDGE)

In [35]:
query_types = ["desc", "geo", "infer"]
reference_dir = "data/correct_results"
generated_dir = "data/generated/%s"
generated_models = os.listdir("data/generated")
generated_models

['Llama 3.1 8B Instruct 128k', 'Llama 3.2 3B Instruct', 'Reasoner v1']

In [36]:
os.makedirs("data/llm_evaluation", exist_ok=True)

In [49]:
for model in generated_models:
    available_query_types = os.listdir(generated_dir % model)
    for query_type in available_query_types:
        generated_folder = os.path.join(generated_dir % model, query_type)
        generated_files = os.listdir(generated_folder)
        reference_folder = os.path.join(reference_dir, query_type)
        reference_files = os.listdir(reference_folder)
        matching_files = sorted(set(generated_files).intersection(reference_files))
        results_folder = os.path.join("data/llm_evaluation", model, query_type)
        os.makedirs(results_folder, exist_ok=True)
        for file in matching_files:
            with open(os.path.join(generated_folder, file), "r") as f:
                generated_code = f.read()
            with open(os.path.join(reference_folder, file), "r") as f:
                reference_code = f.read()
            query_text = query % (reference_code, generated_code)
            response = client_corrector.query(query_text, temperature=0)            
            with open(os.path.join(results_folder, file), "w") as f:
                f.write(response)
    